In [ ]:
from itertools import cycle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, DistributedSampler
from torchvision import datasets, transforms
from model import LeNet5


tran = transforms.Compose((transforms.ToTensor(), transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))))
full_train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=tran)

train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation

train_dataset, val_dataset = random_split(full_train_dataset, (train_size, val_size))

# dloader = DataLoader(train_dataset, 10000, shuffle=True, pin_memory=True)
# print(len(dloader))


In [ ]:
from distributed import distributed_learning

model = distributed_learning(train_dataset, 5, 2, 2, 500, torch.optim.Adam, {"lr": 0.01})

In [ ]:

class PeekableIter:
    def __init__(self, iterable):
        self.iterator = iter(iterable)
        self._next_item = None

    def _advance(self):
        try:
            self._next_item = next(self.iterator)
        except StopIteration:
            self._next_item = None

    def peek(self):
        if self._next_item is None:
            self._advance()
        return self._next_item

    def next(self):
        if self._next_item is None:
            self._advance()
        result = self._next_item
        self._advance()
        return result


class DistributedDataLoader:

    def __init__(self, data, n_workers, local_batch_size):
        self.data = data
        self.n_workers = n_workers
        self.batch_size = local_batch_size

        self.shuffle()

    def shuffle(self):
        self.loaders= [PeekableIter(DataLoader(d, self.batch_size, shuffle=False)) for d in split_N_ways(self.data, self.n_workers)]
        self.need_shuffle: torch.Tensor = torch.zeros(self.n_workers, dtype=bool)

    def set_current_worker(self, current_id):
        self.current_id = current_id

    def get_next_batch(self, id):
        if self.need_shuffle.all():
            self.shuffle()
        elif self.need_shuffle[id]:
            raise Exception("Finish epochs on all workers first") 

        this_batch = self.loaders[id].next()

        next_batch = self.loaders[id].peek()

        if next_batch is None:
            self.need_shuffle[id] = True

        return this_batch
    
    def __next__(self):
        return self.get_next_batch(self.current_id)



In [ ]:
from typing import Iterator
from torch.utils.data import Dataset

class SimpleNumberDataset(Dataset):
    def __init__(self):
        # Input data: numbers from 1 to 10
        self.data = torch.arange(100, dtype=torch.float32)
        self.labels = torch.arange(100, dtype=torch.long) 

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]


def shuffle_and_split(train_data, N, random_seed=None) -> list[Iterator[DataLoader]]:
    if random_seed is not None:
        random_seed = torch.Generator().manual_seed(random_seed)

    lengths = [len(train_data) // N] * N

    return [iter(DataLoader(subset, batch_size=1)) for subset in random_split(train_data, lengths, random_seed)]

d = SimpleNumberDataset()

a = shuffle_and_split(d, 5)



In [ ]:
import matplotlib.pyplot as plt

def plot_metric(metric, **kwargs):
    plt.plot(range(len(metric)), metric, **kwargs)
    plt.xlabel("Epochs")
    plt.grid(True)


plot_metric(train_loss, marker="x")
plot_metric(eval_loss)
plt.ylabel("Loss")